# CS-323 Artificial Intelligence — Complex Engineering Problem
## Intelligent Vacuum-Cleaning Robot

**Group members**

| Roll No. | Name |
|----------|------|
| CS24105 | Sabika Abbas |
| CS24110 | Alishba Javed |
| CS24114 | Bushra Khalid |
| CS24012 | Maryam Siddiqa |

---

**Program flow:** `Schedule.txt` → **CSP** (cleaning order) → `Environment.txt` → **Graph-based search** (navigation) → **Solution**

**What this notebook does**
1. Loads the grid environment from `Environment.txt` and the scheduling constraints from `Schedule.txt` (nothing is hardcoded).
2. Uses a **CSP** (backtracking, MRV + degree, LCV, forward checking) to assign every dirty location to a time slot.
3. Uses **BFS** (uninformed) and **A\*** (informed) to navigate the robot through the cleaning order and back to the charger.
4. Reports the schedule, the path of every journey, costs, nodes expanded, the final grid and the final status.

**How to run:** open this notebook in VS Code, select a Python kernel and use **Run All**.
Keep the notebook and all input files in the **same folder**:
`Environment.txt`, `Schedule.txt`, `Environment_Sample.txt`, `Schedule_Sample.txt`, `Environment_Error.txt`, `Schedule_Error.txt`.

**To test with other input files:** replace `Environment.txt` and `Schedule.txt` with the new files (keep the same file names) and run **Run All** again.
Test 1 always loads whatever `Environment.txt` and `Schedule.txt` it finds, so the schedule, the paths and the costs are computed for the new files.
The sample and error files are used for Tests 2 and 3. **Any other `Environment<name>.txt` + `Schedule<name>.txt` pair in the folder is also run automatically (Test 4), and the summary table at the end lists every pair.**

**Sections:** 0 Imports and configuration · 1 Input loading · 2 Graph-based search · 3 CSP scheduling · 4 Integration · 5 Test runs · 6 Discussion

## 0. Imports and Configuration

In [1]:
import os
import re
import glob
import heapq
from collections import deque

DATA_FOLDER = "inputs"      # folder that contains the input files (falls back to the notebook folder)
ALGORITHM = "A*"            # default navigation algorithm: "BFS" or "A*"

## 1. Input Loading

`load_environment` reads `Environment.txt` (grid size, obstacles, dirty locations, charging station) and validates it:
declared counts, coordinates inside the grid, and no overlap between charger, dirty cells and obstacles.
Dirty locations are labelled **D1, D2, …** in the order they appear in the file.
`unreachable_dirty` flood-fills from the charger and reports dirty cells that cannot be reached.

`load_schedule` reads `Schedule.txt` (number of slots, precedence constraints, latest-slot constraints), and
`validate_schedule_inputs` checks that there are enough slots and that every task label used in a constraint exists.

In [2]:
def parse_tuples(text):
    """'(1,3), (2,4)' -> [(1, 3), (2, 4)]"""
    return [(int(a), int(b)) for a, b in re.findall(r'\(\s*(-?\d+)\s*,\s*(-?\d+)\s*\)', text)]


def load_environment(filepath):
    """
    Load and validate an environment file.
    Returns (grid, charging_station, dirty_locations). Raises ValueError for an invalid file.

    Format (one item or group of items per line):
        rows,cols
        number of obstacles
        (r,c), (r,c), ...        (line present only if the number of obstacles > 0)
        number of dirty locations
        (r,c), (r,c), ...        (line present only if the number of dirty locations > 0)
        (r,c)                    (charging station)
    """
    with open(filepath, 'r', encoding='utf-8-sig') as f:
        lines = [ln.strip() for ln in f if ln.strip()]

    try:
        i = 0
        rows, cols = map(int, lines[i].split(',')); i += 1

        num_obstacles = int(lines[i]); i += 1
        obstacles = []
        if num_obstacles > 0:
            obstacles = parse_tuples(lines[i]); i += 1

        num_dirty = int(lines[i]); i += 1
        dirty = []
        if num_dirty > 0:
            dirty = parse_tuples(lines[i]); i += 1

        cs_list = parse_tuples(lines[i])
    except (IndexError, ValueError):
        raise ValueError("file does not follow the required Environment.txt format")

    # ---- validation ----
    if len(cs_list) != 1:
        raise ValueError("exactly one charging station is required")
    cs = cs_list[0]
    if len(obstacles) != num_obstacles:
        raise ValueError(f"declared {num_obstacles} obstacles but found {len(obstacles)}")
    if len(dirty) != num_dirty:
        raise ValueError(f"declared {num_dirty} dirty locations but found {len(dirty)}")
    all_cells = obstacles + dirty + [cs]
    for (r, c) in all_cells:
        if not (0 <= r < rows and 0 <= c < cols):
            raise ValueError(f"cell {(r, c)} is outside the {rows}x{cols} grid")
    if len(set(all_cells)) != len(all_cells):
        raise ValueError("charging station, dirty locations and obstacles must occupy different cells")

    # ---- build grid ----
    grid = [['.' for _ in range(cols)] for _ in range(rows)]
    for r, c in obstacles:
        grid[r][c] = 'X'
    for k, (r, c) in enumerate(dirty):
        grid[r][c] = f"D{k + 1}"
    grid[cs[0]][cs[1]] = 'C'
    return grid, cs, dirty


def unreachable_dirty(grid, cs, dirty):
    """Flood-fill from the charger; return the dirty locations that cannot be reached."""
    rows, cols = len(grid), len(grid[0])
    seen, queue = {cs}, deque([cs])
    while queue:
        r, c = queue.popleft()
        for dr, dc in ((-1, 0), (1, 0), (0, -1), (0, 1)):
            nr, nc = r + dr, c + dc
            if 0 <= nr < rows and 0 <= nc < cols and grid[nr][nc] != 'X' and (nr, nc) not in seen:
                seen.add((nr, nc))
                queue.append((nr, nc))
    return [(f"D{k + 1}", d) for k, d in enumerate(dirty) if d not in seen]

In [3]:
def slot_to_int(slot):
    """'T3' -> 3"""
    return int(slot[1:])


def load_schedule(filepath):
    """
    Load a scheduling file. Returns (slots, precedence, latest_slot).

    Format (one item per line):
        number of time slots
        number of precedence constraints
        (task_before,task_after)   x number of precedence constraints
        number of latest-slot constraints
        (task,latest_slot)         x number of latest-slot constraints
    """
    with open(filepath, 'r', encoding='utf-8-sig') as f:
        lines = [ln.strip() for ln in f if ln.strip()]

    try:
        i = 0
        num_slots = int(lines[i]); i += 1
        slots = [f"T{k + 1}" for k in range(num_slots)]

        num_prec = int(lines[i]); i += 1
        precedence = []
        for _ in range(num_prec):
            u, v = re.findall(r'[A-Za-z]\w*', lines[i].upper()); i += 1
            precedence.append((u, v))

        num_latest = int(lines[i]); i += 1
        latest_slot = {}
        for _ in range(num_latest):
            task, slot = re.findall(r'[A-Za-z]\w*', lines[i].upper()); i += 1
            # if a task has several latest-slot constraints, the tightest one applies
            if task not in latest_slot or slot_to_int(slot) < slot_to_int(latest_slot[task]):
                latest_slot[task] = slot
    except (IndexError, ValueError):
        raise ValueError("file does not follow the required Schedule.txt format")

    return slots, precedence, latest_slot


def validate_schedule_inputs(variables, slots, precedence, latest_slot):
    """Checks required by the handout. Raises ValueError describing the problem."""
    if len(slots) < len(variables):
        raise ValueError(f"{len(slots)} time slots are not enough for {len(variables)} dirty locations")
    known = set(variables)
    for u, v in precedence:
        for t in (u, v):
            if t not in known:
                raise ValueError(f"constraint refers to unknown task {t} (not a dirty location in the environment)")
    for t, s in latest_slot.items():
        if t not in known:
            raise ValueError(f"constraint refers to unknown task {t} (not a dirty location in the environment)")
        if s not in slots:
            raise ValueError(f"latest-slot constraint uses unknown time slot {s}")

## 2. Graph-Based Search

**Problem formulation**
- **State:** robot position `(row, col)`
- **Initial / goal state:** passed to `search(grid, start, goal, algorithm)` (not hardcoded)
- **Actions:** `UP, DOWN, LEFT, RIGHT`, valid only if the resulting cell is inside the grid and is not an obstacle
- **Transition model:** a valid action moves the robot to the neighbouring cell
- **Step cost:** 1 per move

**Algorithms**
- **BFS (uninformed):** expands nodes level by level. Complete, and optimal because every step costs 1.
- **A\* (informed):** expands the node with the smallest `f(n) = g(n) + h(n)`. Complete, and optimal with an admissible heuristic.

**Heuristic:** Manhattan distance `h(n) = |r − r_goal| + |c − c_goal|`.
- *Admissible:* with 4-directional moves, every path needs at least that many moves (obstacles can only make the real distance longer), so `h` never overestimates.
- *Consistent:* one move changes `h` by at most 1, which equals the step cost, so A\* never needs to re-open a node.

In [4]:
def get_successors(state, grid):
    """Returns [(next_state, action, step_cost), ...] for valid moves (inside the grid, not an obstacle)."""
    r, c = state
    rows, cols = len(grid), len(grid[0])
    moves = [(-1, 0, 'UP'), (1, 0, 'DOWN'), (0, -1, 'LEFT'), (0, 1, 'RIGHT')]
    out = []
    for dr, dc, action in moves:
        nr, nc = r + dr, c + dc
        if 0 <= nr < rows and 0 <= nc < cols and grid[nr][nc] != 'X':
            out.append(((nr, nc), action, 1))
    return out


def goal_test(state, goal_state):
    return state == goal_state


def path_cost(path):
    """Cost = number of moves = len(path) - 1. An empty path or a single state costs 0."""
    if not path:
        return 0
    return len(path) - 1


def heuristic(state, goal_state):
    """
    Manhattan distance.
    Admissible: with 4-directional moves every path needs at least this many moves
    (obstacles can only make the real distance longer), so it never overestimates.
    Consistent: one move changes h by at most 1, which equals the step cost.
    """
    return abs(state[0] - goal_state[0]) + abs(state[1] - goal_state[1])


def validate_path(path, grid):
    """True if consecutive states are adjacent and never on an obstacle."""
    if not path:
        return True
    for k in range(len(path) - 1):
        r1, c1 = path[k]
        r2, c2 = path[k + 1]
        if abs(r1 - r2) + abs(c1 - c2) != 1:
            print(f"  [INVALID] Non-adjacent move {path[k]} -> {path[k + 1]}")
            return False
        if grid[r2][c2] == 'X':
            print(f"  [INVALID] Path enters an obstacle at {path[k + 1]}")
            return False
    return True


def _reconstruct(came_from, node):
    path = []
    while node is not None:
        path.append(node)
        node = came_from[node]
    path.reverse()
    return path

In [5]:
def bfs_search(grid, start, goal):
    """Returns (path, cost, nodes_expanded); path is None if no path exists."""
    if goal_test(start, goal):
        return [start], 0, 0

    frontier = deque([start])
    came_from = {start: None}
    nodes_expanded = 0

    while frontier:
        current = frontier.popleft()
        nodes_expanded += 1
        if goal_test(current, goal):
            path = _reconstruct(came_from, current)
            return path, path_cost(path), nodes_expanded
        for nxt, _, _ in get_successors(current, grid):
            if nxt not in came_from:
                came_from[nxt] = current
                frontier.append(nxt)

    return None, 0, nodes_expanded


def a_star_search(grid, start, goal):
    """Returns (path, cost, nodes_expanded); path is None if no path exists."""
    if goal_test(start, goal):
        return [start], 0, 0

    frontier = [(heuristic(start, goal), 0, start)]      # (f, g, state)
    came_from = {start: None}
    g_costs = {start: 0}
    closed = set()
    nodes_expanded = 0

    while frontier:
        f, g_cur, current = heapq.heappop(frontier)
        if current in closed:                 # outdated queue entry: skip, it is not an expansion
            continue
        closed.add(current)
        nodes_expanded += 1
        if goal_test(current, goal):
            path = _reconstruct(came_from, current)
            return path, path_cost(path), nodes_expanded
        for nxt, _, step in get_successors(current, grid):
            new_g = g_cur + step
            if nxt not in g_costs or new_g < g_costs[nxt]:
                g_costs[nxt] = new_g
                came_from[nxt] = current
                heapq.heappush(frontier, (new_g + heuristic(nxt, goal), new_g, nxt))

    return None, 0, nodes_expanded


def _valid_cell(grid, cell):
    r, c = cell
    return 0 <= r < len(grid) and 0 <= c < len(grid[0]) and grid[r][c] != 'X'


def search(grid, start, goal, algorithm="A*"):
    """
    Dispatcher. Returns (path, cost, nodes_expanded).
    path is None when no path exists (this includes a start/goal that is an obstacle or outside the grid).
    """
    if not _valid_cell(grid, start) or not _valid_cell(grid, goal):
        return None, 0, 0
    alg = algorithm.upper().replace(" ", "")
    if alg == "BFS":
        return bfs_search(grid, start, goal)
    if alg in ("A*", "ASTAR", "A-STAR"):
        return a_star_search(grid, start, goal)
    raise ValueError(f"Unknown algorithm: {algorithm}")

## 3. Constraint Satisfaction Problem (Cleaning-Task Scheduling)

**Formulation**
- **Variables:** `D1 … Dn`, one per dirty location in `Environment.txt` (including tasks that appear in no constraint)
- **Domains:** `{T1 … Tn}` from `Schedule.txt`
- **Constraints:** precedence (`Da` before `Db`), latest slot (`Da` no later than `Tk`), slot exclusivity (at most one task per slot)

**Functions**
- `select_unassigned_variable`: MRV (fewest remaining values); ties are broken by the degree heuristic (most *unassigned* neighbours through constraints).
- `order_domain_values`: Least Constraining Value, so values that remove the fewest options from other variables are tried first.
- `is_consistent`: checks exclusivity, latest-slot and precedence against the current partial assignment.
- `forward_check`: removes infeasible slots from the domains of unassigned variables and returns `None` if a domain becomes empty. It works on a copy of the domains, so removed values are restored automatically on backtracking.
- `backtracking_search`: recursive solver using all of the above. It returns a complete assignment, or `None` if no feasible schedule exists.
- `verify_schedule`: independent check that the generated schedule satisfies every constraint.

In [6]:
def is_consistent(variable, value, assignment, precedence, latest_slot):
    """True if variable=value violates no constraint that can be evaluated with the current assignment."""
    # 1. slot exclusivity
    if value in assignment.values():
        return False
    # 2. latest slot
    if variable in latest_slot and slot_to_int(value) > slot_to_int(latest_slot[variable]):
        return False
    # 3. precedence
    for u, v in precedence:
        if v == variable and u in assignment and slot_to_int(assignment[u]) >= slot_to_int(value):
            return False
        if u == variable and v in assignment and slot_to_int(value) >= slot_to_int(assignment[v]):
            return False
    return True


def forward_check(variable, value, assignment, domains, precedence, latest_slot):
    """
    Prune the domains of unassigned variables after variable=value.
    Returns the new domains, or None if any unassigned variable ends up with an empty domain.
    It works on a copy, so removed values are restored automatically when the search backtracks.
    """
    new_domains = {v: list(d) for v, d in domains.items()}
    for other in domains:
        if other in assignment:
            continue
        # slot exclusivity
        if value in new_domains[other]:
            new_domains[other].remove(value)
        # latest slot
        if other in latest_slot:
            new_domains[other] = [s for s in new_domains[other]
                                  if slot_to_int(s) <= slot_to_int(latest_slot[other])]
        # precedence
        for u, v in precedence:
            if variable == u and other == v:
                new_domains[other] = [s for s in new_domains[other] if slot_to_int(s) > slot_to_int(value)]
            if variable == v and other == u:
                new_domains[other] = [s for s in new_domains[other] if slot_to_int(s) < slot_to_int(value)]
        if not new_domains[other]:
            return None
    return new_domains


def select_unassigned_variable(assignment, domains, precedence):
    """MRV (fewest remaining values); ties broken by the degree heuristic (most unassigned neighbours)."""
    unassigned = [v for v in domains if v not in assignment]
    if not unassigned:
        return None
    min_len = min(len(domains[v]) for v in unassigned)
    candidates = [v for v in unassigned if len(domains[v]) == min_len]
    if len(candidates) == 1:
        return candidates[0]

    def degree(var):
        neighbours = set()
        for u, v in precedence:
            if u == var and v not in assignment:
                neighbours.add(v)
            if v == var and u not in assignment:
                neighbours.add(u)
        return len(neighbours)

    return max(candidates, key=degree)


def order_domain_values(variable, assignment, domains, precedence, latest_slot):
    """Least Constraining Value: values that remove the fewest options from other variables come first."""
    def eliminated(value):
        count = 0
        for other in domains:
            if other == variable or other in assignment:
                continue
            for val in domains[other]:
                if val == value:
                    count += 1
                    continue
                if other in latest_slot and slot_to_int(val) > slot_to_int(latest_slot[other]):
                    count += 1
                    continue
                for u, v in precedence:
                    if variable == u and other == v and slot_to_int(val) <= slot_to_int(value):
                        count += 1
                        break
                    if variable == v and other == u and slot_to_int(val) >= slot_to_int(value):
                        count += 1
                        break
        return count

    return sorted(domains[variable], key=lambda val: (eliminated(val), slot_to_int(val)))

In [7]:
def backtracking_search(variables, domains, precedence, latest_slot):
    """Returns a complete assignment {task: slot}, or None if no feasible schedule exists."""
    def backtrack(assignment, cur_domains):
        if len(assignment) == len(variables):
            return dict(assignment)
        var = select_unassigned_variable(assignment, cur_domains, precedence)
        for value in order_domain_values(var, assignment, cur_domains, precedence, latest_slot):
            if is_consistent(var, value, assignment, precedence, latest_slot):
                assignment[var] = value
                nd = forward_check(var, value, assignment, cur_domains, precedence, latest_slot)
                if nd is not None:
                    result = backtrack(assignment, nd)
                    if result is not None:
                        return result
                del assignment[var]                      # backtrack
        return None

    return backtrack({}, domains)


def verify_schedule(solution, slots, precedence, latest_slot, verbose=True):
    """Independent check that a schedule satisfies every constraint. Returns True if all pass."""
    all_ok = True

    def report(message, passed):
        nonlocal all_ok
        all_ok = all_ok and passed
        if verbose:
            print(f"    [{'PASS' if passed else 'FAIL'}] {message}")

    report("every task has exactly one time slot", all(s in slots for s in solution.values()))
    report("no two tasks share a time slot", len(set(solution.values())) == len(solution))
    for u, v in precedence:
        report(f"{u} before {v}  ({solution[u]} < {solution[v]})",
               slot_to_int(solution[u]) < slot_to_int(solution[v]))
    for t, s in latest_slot.items():
        report(f"{t} no later than {s}  (assigned {solution[t]})",
               slot_to_int(solution[t]) <= slot_to_int(s))
    return all_ok

## 4. Integration

`execute_schedule(grid, charging_station, schedule, algorithm)` takes the CSP-ordered cleaning tasks as `(label, (row, col))` pairs and:
1. starts at the charging station,
2. calls `search` from the robot's current position to each cleaning location in order,
3. appends every returned path without duplicating the shared endpoint, and marks the task as cleaned when it is reached,
4. calls `search` once more to return to the charger,
5. returns `(complete_path, total_cost, success)`.

If a required path cannot be found, it stops and returns `success = False`, so it never claims the task was completed.
`run_pipeline` connects everything (load, CSP, ordering, navigation) and prints all required outputs. `compare_algorithms` runs BFS and A\* on the same pair.

In [8]:
def _execute(grid, charging_station, schedule, algorithm):
    """
    Core routine. Returns (complete_path, total_cost, success, legs, cleaned),
    where legs is a list of per-journey records (used for reporting).
    """
    current = charging_station
    complete_path = [charging_station]
    legs, cleaned = [], []
    targets = list(schedule) + [("Charger", charging_station)]

    for label, goal in targets:
        path, cost, nodes = search(grid, current, goal, algorithm)
        legs.append({'from': current, 'to': goal, 'label': label, 'path': path, 'cost': cost, 'nodes': nodes})

        if path is None:                                   # required path not found -> report failure
            return complete_path, path_cost(complete_path), False, legs, cleaned
        if not validate_path(path, grid):
            legs[-1]['path'] = None
            return complete_path, path_cost(complete_path), False, legs, cleaned

        complete_path.extend(path[1:])                     # skip the duplicated shared endpoint
        current = goal
        if label != "Charger":
            cleaned.append(label)                          # robot reached it -> mark as cleaned

    total_cost = path_cost(complete_path)
    return complete_path, total_cost, True, legs, cleaned


def execute_schedule(grid, charging_station, schedule, algorithm="A*"):
    """
    Executes the CSP-generated schedule using graph-based search.

    schedule : cleaning tasks in slot order, as (label, (row, col)), e.g. [("D3", (4, 1)), ("D1", (2, 4))]
    Returns  : (complete_path, total_cost, success)
    """
    complete_path, total_cost, success, _, _ = _execute(grid, charging_station, schedule, algorithm)
    return complete_path, total_cost, success


def print_grid(grid, title=""):
    if title:
        print(f"  {title}")
    for row in grid:
        print("    " + "  ".join(f"{cell:>2}" for cell in row))


def print_final_grid(grid, cs, complete_path, cleaned):
    """Final grid: robot path marked with *, cleaned dirty cells in lowercase, uncleaned ones unchanged."""
    visited = set(complete_path)
    out = []
    for r in range(len(grid)):
        row = []
        for c in range(len(grid[0])):
            cell = grid[r][c]
            if cell == 'X':
                sym = 'X'
            elif (r, c) == cs:
                sym = 'C'
            elif cell.startswith('D'):
                sym = cell.lower() if cell in cleaned else cell
            elif (r, c) in visited:
                sym = '*'
            else:
                sym = '.'
            row.append(sym)
        out.append(row)
    print_grid(out)
    print("    Legend: C charger | X obstacle | * robot path | d1,d2.. cleaned | D1,D2.. NOT cleaned | . untouched")

In [9]:
def run_pipeline(env_file, schedule_file, algorithm="A*", verbose=True):
    """Complete pipeline for one (environment, schedule) pair. Returns a result dictionary."""
    def say(*args):
        if verbose:
            print(*args)

    say("=" * 70)
    say(f"  PIPELINE: {env_file}  +  {schedule_file}   [{algorithm}]")
    say("=" * 70)

    # ---- 1. load and validate inputs ----
    try:
        grid, cs, dirty = load_environment(env_file)
        slots, precedence, latest_slot = load_schedule(schedule_file)
        dirty_map = {f"D{k + 1}": d for k, d in enumerate(dirty)}
        variables = list(dirty_map)
        validate_schedule_inputs(variables, slots, precedence, latest_slot)
    except FileNotFoundError as e:
        say(f"  [ERROR] File not found: {e.filename}")
        return {"success": False, "reason": f"file not found: {e.filename}"}
    except ValueError as e:
        say(f"  [ERROR] Invalid input: {e}")
        return {"success": False, "reason": f"invalid input: {e}"}

    say(f"  Charging station : {cs}")
    say(f"  Dirty locations  : {dirty_map}")
    say(f"  Time slots       : {slots}")
    say(f"  Precedence       : {precedence}")
    say(f"  Latest slots     : {latest_slot}")
    unreachable = unreachable_dirty(grid, cs, dirty)
    if unreachable:
        say(f"  [WARNING] Unreachable dirty locations: {unreachable}")
    say()
    if verbose:
        print_grid(grid, "BEFORE cleaning (C = charger, D1..Dn = dirty, X = obstacle):")
    say()

    # ---- 2. CSP: generate the cleaning schedule ----
    domains = {v: list(slots) for v in variables}
    solution = backtracking_search(variables, domains, precedence, latest_slot)
    if solution is None:
        say("  [FAILURE] CSP found no feasible schedule - the constraints contradict each other.")
        say("  STATUS: FAILURE - the robot does not move; no dirty location was cleaned.")
        say()
        return {"success": False, "reason": "no feasible schedule exists"}

    order = sorted(solution, key=lambda t: slot_to_int(solution[t]))
    schedule = [(t, dirty_map[t]) for t in order]
    say("  Cleaning schedule generated by the CSP:")
    for t in order:
        say(f"    {solution[t]} -> {t} at {dirty_map[t]}")
    say("  Constraint verification:")
    if verbose:
        verify_schedule(solution, slots, precedence, latest_slot)
    say()

    # ---- 3. navigation ----
    path, cost, ok, legs, cleaned = _execute(grid, cs, schedule, algorithm)
    say(f"  Dirty locations visited, in order: {' -> '.join(cleaned) if cleaned else '(none)'}")
    say()
    say("  Journeys:")
    total_nodes = 0
    for leg in legs:
        total_nodes += leg['nodes']
        if leg['path'] is None:
            say(f"    {leg['from']} -> {leg['label']} {leg['to']}: NO PATH FOUND (nodes expanded: {leg['nodes']})")
        else:
            say(f"    {leg['from']} -> {leg['label']} {leg['to']}: cost {leg['cost']}, nodes expanded {leg['nodes']}")
            say(f"       path: {leg['path']}")
    say()
    if verbose:
        print("  Final grid:")
        print_final_grid(grid, cs, path, cleaned)
    say()
    say(f"  Complete path    : {path}")
    say(f"  Total movements  : {len(path) - 1}")
    say(f"  Total cost       : {cost}")
    say(f"  Nodes expanded   : {total_nodes}")
    if ok:
        say("  STATUS: SUCCESS - all dirty locations cleaned and the robot returned to the charging station.")
    else:
        missing = [t for t in order if t not in cleaned]
        say(f"  STATUS: FAILURE - task NOT completed. Cleaned: {cleaned if cleaned else 'none'} | Not cleaned: {missing}")
    say()

    return {"success": ok, "reason": "OK" if ok else "a required path was not found",
            "solution": solution, "schedule": schedule, "path": path, "cost": cost,
            "moves": len(path) - 1, "nodes": total_nodes, "cleaned": cleaned}


def compare_algorithms(env_file, schedule_file):
    """Run BFS and A* on the same pair and print a comparison table."""
    print("=" * 70)
    print(f"  ALGORITHM COMPARISON: {os.path.basename(env_file)} + {os.path.basename(schedule_file)}")
    print("=" * 70)
    print(f"  {'Algorithm':<11}{'Success':<10}{'Cost':<8}{'Movements':<12}{'Nodes expanded':<15}")
    print("  " + "-" * 56)
    for alg in ("BFS", "A*"):
        r = run_pipeline(env_file, schedule_file, alg, verbose=False)
        if "cost" in r:
            print(f"  {alg:<11}{str(r['success']):<10}{r['cost']:<8}{r['moves']:<12}{r['nodes']:<15}")
        else:
            print(f"  {alg:<11}{'False':<10}{'-':<8}{'-':<12}{'-':<15}({r['reason']})")
    print()


def find_pairs(folder):
    """Match Environment<suffix>.txt with Schedule<suffix>.txt inside the folder."""
    pairs = []
    for env in sorted(glob.glob(os.path.join(folder, "Environment*.txt"))):
        suffix = os.path.basename(env)[len("Environment"):-len(".txt")]
        sched = os.path.join(folder, f"Schedule{suffix}.txt")
        if os.path.exists(sched):
            pairs.append((env, sched))
    return pairs


def resolve_folder(folder=DATA_FOLDER):
    """Folder holding the input files: the notebook folder if it contains Environment*.txt files, else the data folder."""
    if glob.glob("Environment*.txt"):
        return "."
    if glob.glob(os.path.join(folder, "Environment*.txt")):
        return folder
    return "."


def locate(name):
    """Path of an input file: next to the notebook first, then inside the data folder."""
    for candidate in (name, os.path.join(DATA_FOLDER, name)):
        if os.path.exists(candidate):
            return candidate
    return name


def run_all_tests(algorithm="A*", folder=DATA_FOLDER):
    """Run every Environment*/Schedule* pair found in the folder, with a BFS vs A* comparison for each."""
    folder = resolve_folder(folder)
    pairs = find_pairs(folder)
    if not pairs:
        print(f"No matching Environment*.txt / Schedule*.txt pairs found in '{folder}'.")
        return
    print(f"Found {len(pairs)} test pair(s) in '{folder}':")
    for e, s in pairs:
        print(f"   - {os.path.basename(e)}  +  {os.path.basename(s)}")
    print()

    results = []
    for e, s in pairs:
        results.append((e, s, run_pipeline(e, s, algorithm)))
        compare_algorithms(e, s)

    print("=" * 70)
    print("  SUMMARY")
    print("=" * 70)
    for e, s, r in results:
        status = "SUCCESS" if r["success"] else "FAILED"
        detail = f"cost={r['cost']}" if r["success"] else r["reason"]
        print(f"  {os.path.basename(e):<24}+ {os.path.basename(s):<22}-> {status} ({detail})")

## 5. Test Runs

In [10]:
def f(name):
    # path of an input file: next to the notebook first, then inside the data folder
    return locate(name)

print("Test 1 will use these input files:")
print("   ", f("Environment.txt"))
print("   ", f("Schedule.txt"))

Test 1 will use these input files:
    inputs\Environment.txt
    inputs\Schedule.txt


### Test 0 — Search edge cases
Start equals goal, unreachable goal (D1 is walled in inside `Environment_Error.txt`), and invalid start/goal.

In [11]:
grid, cs, dirty = load_environment(f("Environment.txt"))
print("start == goal         :", search(grid, cs, cs, "BFS"), "|", search(grid, cs, cs, "A*"))
print("invalid start (wall)  :", search(grid, (1, 3), dirty[0], "A*"))
print("goal outside the grid :", search(grid, cs, (99, 99), "BFS"))
print("path_cost([]) =", path_cost([]), "| path_cost([(0, 0)]) =", path_cost([(0, 0)]))

egrid, ecs, edirty = load_environment(f("Environment_Error.txt"))
for alg in ("BFS", "A*"):
    path, cost, nodes = search(egrid, ecs, edirty[0], alg)
    print(f"unreachable D1 {edirty[0]} with {alg:<3}: path={path}, cost={cost}, nodes expanded={nodes}")

start == goal         : ([(3, 0)], 0, 0) | ([(3, 0)], 0, 0)
invalid start (wall)  : (None, 0, 0)
goal outside the grid : (None, 0, 0)
path_cost([]) = 0 | path_cost([(0, 0)]) = 0
unreachable D1 (2, 4) with BFS: path=None, cost=0, nodes expanded=53
unreachable D1 (2, 4) with A* : path=None, cost=0, nodes expanded=53


### Test 1 — Own environment (`Environment.txt` + `Schedule.txt`)

In [12]:
run_pipeline(f("Environment.txt"), f("Schedule.txt"), "A*")
compare_algorithms(f("Environment.txt"), f("Schedule.txt"))

  PIPELINE: inputs\Environment.txt  +  inputs\Schedule.txt   [A*]
  Charging station : (3, 0)
  Dirty locations  : {'D1': (2, 4), 'D2': (5, 6), 'D3': (4, 1), 'D4': (6, 3)}
  Time slots       : ['T1', 'T2', 'T3', 'T4']
  Precedence       : [('D3', 'D1'), ('D1', 'D2'), ('D3', 'D4')]
  Latest slots     : {'D2': 'T3', 'D4': 'T4'}

  BEFORE cleaning (C = charger, D1..Dn = dirty, X = obstacle):
     .   .   .   .   .   .   .   .
     .   .   .   X   .   .   X   .
     .   .   .   .  D1   .   .   .
     C   .   .   .   .   .   .   .
     .  D3   .   .   X   .   .   .
     .   .   X   .   .   .  D2   .
     X   .   .  D4   .   X   .   .
     .   .   .   .   .   .   .   .

  Cleaning schedule generated by the CSP:
    T1 -> D3 at (4, 1)
    T2 -> D1 at (2, 4)
    T3 -> D2 at (5, 6)
    T4 -> D4 at (6, 3)
  Constraint verification:
    [PASS] every task has exactly one time slot
    [PASS] no two tasks share a time slot
    [PASS] D3 before D1  (T1 < T2)
    [PASS] D1 before D2  (T2 < T3)
    [P

### Test 2 — Provided sample (`Environment_Sample.txt` + `Schedule_Sample.txt`)

In [13]:
run_pipeline(f("Environment_Sample.txt"), f("Schedule_Sample.txt"), "A*")
compare_algorithms(f("Environment_Sample.txt"), f("Schedule_Sample.txt"))

  PIPELINE: inputs\Environment_Sample.txt  +  inputs\Schedule_Sample.txt   [A*]
  Charging station : (0, 0)
  Dirty locations  : {'D1': (0, 7), 'D2': (3, 7), 'D3': (7, 7), 'D4': (7, 0)}
  Time slots       : ['T1', 'T2', 'T3', 'T4']
  Precedence       : [('D1', 'D3'), ('D2', 'D4'), ('D1', 'D4')]
  Latest slots     : {'D3': 'T3', 'D2': 'T3'}

  BEFORE cleaning (C = charger, D1..Dn = dirty, X = obstacle):
     C   .   .   .   .   .   .  D1
     .   .   .   X   .   .   .   .
     .   .   .   X   .   .   .   .
     .   .   .   X   .   .   .  D2
     .   .   .   X   .   .   .   .
     .   .   .   X   .   .   .   .
     .   .   .   X   .   .   .   .
    D4   .   .   .   .   .   .  D3

  Cleaning schedule generated by the CSP:
    T1 -> D1 at (0, 7)
    T2 -> D2 at (3, 7)
    T3 -> D3 at (7, 7)
    T4 -> D4 at (7, 0)
  Constraint verification:
    [PASS] every task has exactly one time slot
    [PASS] no two tasks share a time slot
    [PASS] D1 before D3  (T1 < T3)
    [PASS] D2 before D4  (T

### Test 3 — Failure cases

**3a. Infeasible schedule** (`Environment_Error.txt` + `Schedule_Error.txt`): the precedence constraints form a cycle (D1 → D2 → D3 → D1), so the CSP reports failure and the robot does not move.

In [14]:
run_pipeline(f("Environment_Error.txt"), f("Schedule_Error.txt"), "A*")

  PIPELINE: inputs\Environment_Error.txt  +  inputs\Schedule_Error.txt   [A*]
  Charging station : (3, 0)
  Dirty locations  : {'D1': (2, 4), 'D2': (5, 6), 'D3': (4, 1), 'D4': (6, 3)}
  Time slots       : ['T1', 'T2', 'T3', 'T4']
  Precedence       : [('D1', 'D2'), ('D2', 'D3'), ('D3', 'D1')]
  Latest slots     : {'D4': 'T1'}
  [WARNING] Unreachable dirty locations: [('D1', (2, 4))]

  BEFORE cleaning (C = charger, D1..Dn = dirty, X = obstacle):
     .   .   .   .   .   .   .   .
     .   .   .   X   X   .   X   .
     .   .   .   X  D1   X   .   .
     C   .   .   .   X   .   .   .
     .  D3   .   .   X   .   .   .
     .   .   X   .   .   .  D2   .
     X   .   .  D4   .   X   .   .
     .   .   .   .   .   .   .   .

  [FAILURE] CSP found no feasible schedule - the constraints contradict each other.
  STATUS: FAILURE - the robot does not move; no dirty location was cleaned.



{'success': False, 'reason': 'no feasible schedule exists'}

**3b. Unreachable destination** (`Environment_Error.txt` + `Schedule.txt`): the schedule is feasible, but D1 is walled in, so graph search cannot reach it and the integration reports failure instead of claiming success.

In [15]:
run_pipeline(f("Environment_Error.txt"), f("Schedule.txt"), "A*")
compare_algorithms(f("Environment_Error.txt"), f("Schedule.txt"))

  PIPELINE: inputs\Environment_Error.txt  +  inputs\Schedule.txt   [A*]
  Charging station : (3, 0)
  Dirty locations  : {'D1': (2, 4), 'D2': (5, 6), 'D3': (4, 1), 'D4': (6, 3)}
  Time slots       : ['T1', 'T2', 'T3', 'T4']
  Precedence       : [('D3', 'D1'), ('D1', 'D2'), ('D3', 'D4')]
  Latest slots     : {'D2': 'T3', 'D4': 'T4'}
  [WARNING] Unreachable dirty locations: [('D1', (2, 4))]

  BEFORE cleaning (C = charger, D1..Dn = dirty, X = obstacle):
     .   .   .   .   .   .   .   .
     .   .   .   X   X   .   X   .
     .   .   .   X  D1   X   .   .
     C   .   .   .   X   .   .   .
     .  D3   .   .   X   .   .   .
     .   .   X   .   .   .  D2   .
     X   .   .  D4   .   X   .   .
     .   .   .   .   .   .   .   .

  Cleaning schedule generated by the CSP:
    T1 -> D3 at (4, 1)
    T2 -> D1 at (2, 4)
    T3 -> D2 at (5, 6)
    T4 -> D4 at (6, 3)
  Constraint verification:
    [PASS] every task has exactly one time slot
    [PASS] no two tasks share a time slot
    [PASS] D

### Test 4 — Any other input pairs found in the folder
Every other `Environment<name>.txt` + `Schedule<name>.txt` pair that is present next to the notebook is run automatically
(the suffix must match, for example `Environment_Test.txt` with `Schedule_Test.txt`). Files without a matching partner are reported.
The pairs already covered by Tests 1 to 3 are not repeated here.

In [16]:
folder = resolve_folder(DATA_FOLDER)
standard = {"Environment.txt", "Environment_Sample.txt", "Environment_Error.txt"}

def file_suffixes(prefix):
    names = glob.glob(os.path.join(folder, prefix + "*.txt"))
    return {os.path.basename(n)[len(prefix):-len(".txt")] for n in names}

for sfx in sorted(file_suffixes("Environment") - file_suffixes("Schedule")):
    print(f"[WARNING] Environment{sfx}.txt has no matching Schedule{sfx}.txt, so it was not run.")
for sfx in sorted(file_suffixes("Schedule") - file_suffixes("Environment")):
    print(f"[WARNING] Schedule{sfx}.txt has no matching Environment{sfx}.txt, so it was not run.")

extra_pairs = [(e, s) for e, s in find_pairs(folder) if os.path.basename(e) not in standard]
if not extra_pairs:
    print("No additional Environment/Schedule pairs found. Tests 1 to 3 cover all files in the folder.")
for e, s in extra_pairs:
    run_pipeline(e, s, ALGORITHM)
    compare_algorithms(e, s)

  PIPELINE: inputs\Environment_Test1.txt  +  inputs\Schedule_Test1.txt   [A*]
  Charging station : (4, 4)
  Dirty locations  : {'D1': (0, 9), 'D2': (5, 5), 'D3': (9, 0), 'D4': (0, 0), 'D5': (9, 9)}
  Time slots       : ['T1', 'T2', 'T3', 'T4', 'T5', 'T6']
  Precedence       : [('D1', 'D2'), ('D3', 'D4')]
  Latest slots     : {'D2': 'T4', 'D5': 'T6'}

  BEFORE cleaning (C = charger, D1..Dn = dirty, X = obstacle):
    D4   .   .   .   .   .   .   .   .  D1
     .   .   .   .   .   .   .   .   .   .
     .   .   X   .   .   .   .   .   .   .
     .   .   .   .   .   .   .   .   .   .
     .   .   .   .   C   .   .   .   .   .
     .   .   .   .   .  D2   .   .   .   .
     .   .   .   .   .   .   .   .   .   .
     .   .   .   .   .   .   .   X   .   .
     .   .   .   .   .   .   .   .   .   .
    D3   .   .   .   .   .   .   .   .  D5

  Cleaning schedule generated by the CSP:
    T1 -> D1 at (0, 9)
    T2 -> D2 at (5, 5)
    T3 -> D3 at (9, 0)
    T4 -> D4 at (0, 0)
    T5 -> D5 at (9,

### Summary of all runs

In [17]:
# every Environment/Schedule pair found in the folder, plus the mixed failure case of Test 3b
all_runs = list(find_pairs(folder))
all_runs.append((f("Environment_Error.txt"), f("Schedule.txt")))

print(f"{'Pair':<46}{'Algo':<6}{'Success':<9}{'Cost':<7}{'Nodes':<7}Note")
print("-" * 94)
for e, s in all_runs:
    name = f"{os.path.basename(e)} + {os.path.basename(s)}"
    for alg in ("BFS", "A*"):
        r = run_pipeline(e, s, alg, verbose=False)
        if "cost" in r:
            note = "" if r["success"] else r["reason"]
            print(f"{name:<46}{alg:<6}{str(r['success']):<9}{r['cost']:<7}{r['nodes']:<7}{note}")
        else:
            print(f"{name:<46}{alg:<6}{'False':<9}{'-':<7}{'-':<7}{r['reason']}")

Pair                                          Algo  Success  Cost   Nodes  Note
----------------------------------------------------------------------------------------------
Environment.txt + Schedule.txt                BFS   True     22     138    
Environment.txt + Schedule.txt                A*    True     22     46     
Environment_Error.txt + Schedule_Error.txt    BFS   False    -      -      no feasible schedule exists
Environment_Error.txt + Schedule_Error.txt    A*    False    -      -      no feasible schedule exists
Environment_Sample.txt + Schedule_Sample.txt  BFS   True     28     118    
Environment_Sample.txt + Schedule_Sample.txt  A*    True     28     33     
Environment_Test1.txt + Schedule_Test1.txt    BFS   True     64     443    
Environment_Test1.txt + Schedule_Test1.txt    A*    True     64     233    
Environment_Test2.txt + Schedule_Test2.txt    BFS   True     40     171    
Environment_Test2.txt + Schedule_Test2.txt    A*    True     40     111    
Environment

### Optional: interactive mode (file picker)
Set `INTERACTIVE_MODE = True` and run this cell to choose any `Environment*.txt` file with a dialog window.
The matching `Schedule*.txt` (same suffix, e.g. `Environment_Test.txt` and `Schedule_Test.txt`) is found automatically.
It is off by default so that **Run All** never waits for a popup (and it still works on machines without a display).

In [18]:
INTERACTIVE_MODE = False     # True -> open a file dialog to choose an Environment file

if INTERACTIVE_MODE:
    selected_env = ""
    try:
        import tkinter as tk
        from tkinter import filedialog
        root = tk.Tk()
        root.withdraw()                         # hide the empty main window
        root.attributes('-topmost', True)       # keep the dialog on top of other windows
        root.update()
        selected_env = filedialog.askopenfilename(
            title="Select Environment File",
            initialdir=os.path.abspath(resolve_folder()),
            filetypes=[("Text Files", "*.txt")])
        root.destroy()
    except Exception as e:
        print("File dialog is not available here:", e)

    if not selected_env:
        print("No file selected (cancelled).")
    else:
        env_name = os.path.basename(selected_env)
        if not (env_name.startswith("Environment") and env_name.endswith(".txt")):
            print("Please select a file named Environment*.txt")
        else:
            suffix = env_name[len("Environment"):-len(".txt")]
            selected_sched = os.path.join(os.path.dirname(selected_env), f"Schedule{suffix}.txt")
            print("Selected environment :", env_name)
            print("Matching schedule    :", os.path.basename(selected_sched))
            if not os.path.exists(selected_sched):
                print("[ERROR] Matching schedule file not found. Both files must share the same suffix.")
            else:
                run_pipeline(selected_env, selected_sched, ALGORITHM)
                compare_algorithms(selected_env, selected_sched)
else:
    print("Interactive mode is off. Set INTERACTIVE_MODE = True to pick an Environment file with a dialog.")

Interactive mode is off. Set INTERACTIVE_MODE = True to pick an Environment file with a dialog.


## 6. Discussion

- **Correctness:** every path is checked with `validate_path` (adjacent cells, no obstacles) and every schedule with `verify_schedule`. Failures are reported explicitly: the CSP reports an infeasible schedule, and `execute_schedule` reports a missing path with `success = False`.
- **BFS:** complete and optimal for unit step costs, but uninformed, so it expands nodes in every direction.
- **A\*:** complete and optimal with the admissible and consistent Manhattan heuristic. It returns the same cost as BFS and usually expands far fewer nodes. When the goal is unreachable it must still exhaust the reachable region, so it has no advantage there. A long wall in the straight line to the goal also reduces its advantage.
- **CSP:** backtracking with MRV, degree, LCV and forward checking is complete, so it correctly reports infeasible schedules such as a precedence cycle.
- **Limitations:** the cleaning order is fixed by the CSP and is not chosen to minimise travel distance, so each journey is optimal but the whole tour need not be. Grid costs are uniform.
- **Possible improvements:** treat the visiting order as an optimisation problem (add travel cost to the CSP, or search over all feasible orders), add arc consistency (AC-3), support weighted cells, and use faster search such as bidirectional search for large grids.

## References
- Russell, S. and Norvig, P., *Artificial Intelligence: A Modern Approach* (graph search, A\*, constraint satisfaction problems).
- CS-323 course material and the CEP problem description.